# Visual Area Identification in M132 Parcellation

## Objective
Identify which areas in the **M132 brain parcellation** correspond to **visual areas** by analyzing their overlap with known visual regions from the **FVE/FVE_all parcellation**.

## Methodology

### Data Structure
The provided JSON data represents **overlap percentages** between two parcellation schemes:
- **M132 areas** (e.g., `V1_M132`, `V2_M132`, `MT_M132`)
- **FVE/FVE_all areas** (e.g., `FVE.V1`, `FVE_all.V2`, `FVE.MT`)

Each entry shows: `M132_area: {FVE_area: percentage, ...}` where the percentage indicates how much of that FVE area overlaps with the M132 area.

### Visual Area List
We have a predefined list of **32 visual areas** from the FVE parcellation:
```python
area_list = ['V1', 'V2', 'VP', 'V3', 'PIP', 'V3A', 'MT', 'V4t', 'V4',
             'PO', 'VOT', 'DP', 'MIP', 'MDP', 'MSTd', 'VIP', 'LIP',
             'PITv', 'PITd', 'AITv', 'MSTl', 'FST', 'CITv', 'CITd',
             '7a', 'STPp', 'STPa', 'FEF', '46', 'TF', 'TH', 'AITd']
```

### Analysis Approach

We implemented a **multi-faceted analytical approach**:

#### 1. **Direct Overlap Quantification**
For each M132 area, we calculate what percentage of its total FVE overlap comes from known visual areas:

```
Visual Overlap % = (Sum of overlaps with visual areas / Total overlap with all areas) × 100
```

#### 2. **Name-Based Pattern Recognition**
We incorporated domain knowledge by looking for **visual area patterns** in M132 area names:
- `V1`, `V2`, `V3`, `V4`, `V6` → Primary/secondary visual cortices
- `MT`, `MST` → Motion-sensitive areas
- `PIT`, `AIT`, `CIT` → Temporal visual areas
- `DP`, `PO` → Dorsal/parietal visual areas

#### 3. **Confidence Scoring System**
We calculate a **weighted confidence score** combining:
- **70% weight**: Direct overlap percentage (quantitative evidence)
- **30% weight**: Name pattern match (qualitative/domain knowledge)

```
Confidence Score = (Overlap% × 0.7) + (NamePattern × 0.3)
```

#### 4. **Threshold-Based Classification**
We apply the following classification rules:
- **High Confidence**: ≥50% visual overlap + confidence score ≥0.5
- **Borderline**: 20-49% visual overlap (may require manual inspection)
- **Non-Visual**: <20% visual overlap

## Key Findings

### Expected High-Confidence Visual Areas
Areas with **M132 nomenclature directly indicating visual function** will naturally score highest:
- `V1_M132`, `V2_M132`, `V3_M132`, `V4_M132` → Early visual cortices
- `MT_M132` → Middle temporal/VS/MT+ complex
- `V6_M132`, `V6A_M132` → Parietal visual areas

### Discovery of "Hidden" Visual Areas
The analysis helps identify **areas without obvious visual names** that nevertheless have substantial visual overlap:
- Areas with significant overlap with `V3A`, `PIP`, `DP`, etc.
- Areas showing strong connections to dorsal/ventral visual streams

### Quantitative vs. Qualitative Evidence
Some interesting cases emerge:
1. **High overlap, non-visual name** → Likely visual area with different nomenclature
2. **Low overlap, visual-sounding name** → Possibly misnamed or has multiple functions
3. **Mixed overlap profile** → May represent transitional/border regions

## Interpretation Guidelines

### When to Include an Area
Consider including M132 areas when they show:
1. **≥50% visual overlap** AND
2. **Confidence score ≥0.5** OR
3. **Clear anatomical/functional evidence** despite lower percentages

### Caution with Borderline Cases
Areas with **20-50% visual overlap** may represent:
- **Multimodal areas** with both visual and non-visual functions
- **Transition zones** between visual and adjacent regions
- **Individual variability** in parcellation boundaries



In [2]:
import numpy as np
import os
import json

In [3]:
"""
Set input and output paths
"""
basepath = os.path.abspath(os.path.join(os.path.dirname('')))
datapath = os.path.join(basepath, '')
out_label = ''
out_path = basepath


FV91_overlap_areas = ['1', '10', '11', '12', '13', '14', '2', '23', '24', '25', 
                      '29', '30', '32', '35', '36', '3a', '3b', '4', '45', '46', 
                      '5', '6', '7a', '7b', '9', 'A1', 'AI', 'AITd', 'AITv', 'AUD', 
                      'Aud', 'CA1', 'CA3', 'CITd', 'CITv', 'CM', 'DP', 'ER', 'FEF', 
                      'FST', 'G', 'Id', 'Ig', 'L', 'LIP', 'MDP', 'MIP', 'MSTd', 'MSTl', 
                      'MT', 'PA', 'PAC', 'PIP', 'PIR', 'PITd', 'PITv', 'PO', 'PS', 'PSd', 
                      'PaS', 'Pall', 'PrS', 'Pro', 'RL', 'Ri', 'S', 'SII', 'SMA', 'STPa', 
                      'STPp', 'TF', 'TH', 'UNK', 'V1', 'V2', 'V3', 'V3A', 'V4', 'V4t', 
                      'VIP', 'VOT', 'VP']


visual_areas =  ['V1', 'V2', 'VP', 'V3', 'PIP', 'V3A', 'MT', 'V4t', 'V4',
                 'PO', 'VOT', 'DP', 'MIP', 'MDP', 'MSTd', 'VIP', 'LIP',
                 'PITv', 'PITd', 'AITv', 'MSTl', 'FST', 'CITv', 'CITd',
                 '7a', 'STPp', 'STPa', 'FEF', '46', 'TF', 'TH', 'AITd']



# Load overlap data with avg normalization for FVE.all
f = open(os.path.join(datapath, 'M132_FV91all_avg_norm.json'), 'r')
fve_all_overlap_avg_norm = json.load(f)
f.close()

# Load overlap data with avg normalization for FVE
g = open(os.path.join(datapath, 'M132_FV91_avg_norm.json'), 'r')
fve_overlap_avg_norm = json.load(g)
g.close()

# Load overlap data with row normalization for FVE.all
A = open(os.path.join(datapath, 'M132_FV91all_row_normalize.json'), 'r')
fve_all_overlap_row_norm = json.load(A)
A.close()

# Load overlap data with row normalization for FVE
B = open(os.path.join(datapath, 'M132_FV91_row_normalize.json'), 'r')
fve_overlap_row_norm = json.load(B)
B.close()

Area = 'PBr_M132'
print('FVE_all row normalize =', sum(list(fve_all_overlap_row_norm[Area].values())))
print('FVE row normalize =', sum(list(fve_overlap_row_norm[Area].values())))
print('FVE_all avg normalize =', sum(list(fve_all_overlap_avg_norm[Area].values())))
print('FVE avg normalize =', sum(list(fve_overlap_avg_norm[Area].values())))


FVE_all row normalize = 100.1
FVE row normalize = 100.0
FVE_all avg normalize = 58.3
FVE avg normalize = 14.9


In [4]:
fve_all_overlap_row_norm[Area]

{'FVE_all.35': 1.4,
 'FVE_all.36': 17,
 'FVE_all.UNK': 29.6,
 'FVE.36': 0.3,
 'FVE_all.STPa': 6.6,
 'UNK': 7.4,
 'FVE.STPa': 6.6,
 'FVE_all.STPp': 23,
 'FVE.STPp': 4.9,
 'FVE.AUD': 0.3,
 'FVE_all.Aud': 3}

In [5]:
fve_overlap_row_norm['F5_M132']

{'???': 99.8, 'FVE.FEF': 0.2}

In [6]:
fve_overlap_row_norm[Area]

{'???': 70.7, 'FVE.STPa': 6.3, 'FVE.STPp': 23}

## M132 visual areas analysis

In [7]:
def advanced_visual_analysis(data, visual_areas):
    """
    More sophisticated analysis incorporating domain knowledge
    """
    # Known visual area patterns in M132 nomenclature
    visual_patterns = ['V1', 'V2', 'VP', 'V3', 'PIP', 'V3A', 'MT', 'V4t', 'V4',
                 'PO', 'VOT', 'DP', 'MIP', 'MDP', 'MSTd', 'VIP', 'LIP',
                 'PITv', 'PITd', 'AITv', 'MSTl', 'FST', 'CITv', 'CITd',
                 '7a', 'STPp', 'STPa', 'FEF', '46', 'TF', 'TH', 'AITd']

    
    results = {}
    
    for m132_area, fve_overlaps in data.items():
        # Method 1: Direct overlap percentage
        total_percentage = sum(fve_overlaps.values())
        visual_percentage = 0
        visual_contributors = []
        
        for fve_area, percentage in fve_overlaps.items():
            base_area = fve_area.split('.', 1)[1] if '.' in fve_area else fve_area
            if base_area in visual_areas:
                visual_percentage += percentage
                visual_contributors.append((base_area, percentage))
        
        # Method 2: Name-based classification
        name_based_score = 0
        for pattern in visual_patterns:
            if pattern in m132_area:
                name_based_score = 1
                break
        
        # Method 3: Major contributor analysis
        major_visual_contributor = None
        if visual_contributors:
            major_visual_contributor = max(visual_contributors, key=lambda x: x[1])
        
        overlap_ratio = (visual_percentage / total_percentage * 100) if total_percentage > 0 else 0
        
        results[m132_area] = {
            "overlap_percentage": overlap_ratio,
            "total_visual_overlap": visual_percentage,
            "name_suggests_visual": name_based_score,
            "major_visual_contributor": major_visual_contributor,
            "confidence_score": (overlap_ratio/100 * 0.7 + name_based_score * 0.3)  # Weighted score
            #"confidence_score": (overlap_ratio/100)
        }
    
    return results

# Run advanced analysis
advanced_results = advanced_visual_analysis(fve_overlap_row_norm, visual_areas)

# Sort by confidence score
sorted_advanced = sorted(advanced_results.items(), 
                        key=lambda x: x[1]["confidence_score"], reverse=True)
List = []
print("\nAdvanced Visual Area Classification (Sorted by Confidence):")
print("=" * 70)
for area, info in sorted_advanced[:]:
    List.append(area)
    major = info['major_visual_contributor']
    major_str = f"{major[0]}({major[1]:.1f}%)" if major else "None"
    print(f"{area:<20} {info['confidence_score']:5.2f} "
          f"Overlap: {info['overlap_percentage']:5.1f}% "
          f"Major: {major_str}")
#print("=" * 70)
#for area, info in sorted_advanced[25:41]:
#    major = info['major_visual_contributor']
#    major_str = f"{major[0]}({major[1]:.1f}%)" if major else "None"
#    print(f"{area:<20} {info['confidence_score']:5.2f} "
#          f"Overlap: {info['overlap_percentage']:5.1f}% "
#          f"Major: {major_str}")


Advanced Visual Area Classification (Sorted by Confidence):
PO_M132               1.00 Overlap: 100.0% Major: PO(100.0%)
V4t_M132              1.00 Overlap: 100.0% Major: V4(99.3%)
DP_M132               1.00 Overlap:  99.4% Major: DP(61.1%)
V4_M132               0.99 Overlap:  98.7% Major: V4(61.8%)
V3A_M132              0.99 Overlap:  98.0% Major: V3A(50.2%)
V3_M132               0.98 Overlap:  97.4% Major: V2(47.9%)
V1_M132               0.98 Overlap:  97.3% Major: V1(91.8%)
MT_M132               0.97 Overlap:  95.1% Major: MT(28.2%)
V2_M132               0.97 Overlap:  95.1% Major: V2(78.4%)
PIP_M132              0.95 Overlap:  92.3% Major: DP(39.5%)
FST_M132              0.92 Overlap:  88.5% Major: PITd(43.3%)
TH_TF_M132            0.92 Overlap:  88.4% Major: TF(64.9%)
46v_M132              0.82 Overlap:  74.9% Major: 46(74.9%)
9_46v_M132            0.81 Overlap:  73.3% Major: 46(73.3%)
LIP_M132              0.80 Overlap:  72.1% Major: VIP(36.4%)
9_46d_M132            0.79 Overlap

In [8]:
print(List)

['PO_M132', 'V4t_M132', 'DP_M132', 'V4_M132', 'V3A_M132', 'V3_M132', 'V1_M132', 'MT_M132', 'V2_M132', 'PIP_M132', 'FST_M132', 'TH_TF_M132', '46v_M132', '9_46v_M132', 'LIP_M132', '9_46d_M132', '46d_M132', 'TEOm_M132', 'TEpd_M132', 'STPi_M132', 'TEO_M132', 'MST_M132', 'STPc_M132', 'V6_M132', 'TEpv_M132', 'TEam-p_M132', '8l_M132', 'VIP_M132', '8r_M132', 'V6A_M132', 'PGa_M132', 'IPa_M132', '45B_M132', '7A_M132', 'TEad_M132', '8m_M132', 'PBc_M132', 'TEam-a_M132', 'MIP_M132', 'STPr_M132', '???', 'PBr_M132', 'Prostriate_M132', '44_M132', 'TEav_M132', '7m_M132', '8B_M132', 'TPt_M132', 'Subiculum_M132', '45A_M132', 'LB_M132', '13_M132', 'Perirhinal_M132', '5_M132', '23_M132', 'F5_M132', '3_M132', '12_M132', 'MEDIAL.WALL', '29_30_M132', 'Entorhinal_M132', '31_M132', '24a_M132', '24b_M132', '1_M132', '2_M132', '24d_M132', '25_M132', 'Temporal-pole_M132', 'F1_M132', '14_M132', '24c_M132', 'OPAI_M132', '32_M132', 'Insula_M132', 'F3_M132', 'OPRO_M132', '10_M132', 'Piriform_M132', 'F6_M132', 'Pi_M132

In [10]:
def get_recommended_visual_m132_areas(data, visual_areas, confidence_threshold=0.20):
    """
    Get final recommended list of M132 visual areas
    """
    advanced_results = advanced_visual_analysis(data, visual_areas)
    
    recommended_areas = []
    for m132_area, info in advanced_results.items():
        if (info['confidence_score'] >= confidence_threshold and 
            info['overlap_percentage'] >= 30):
            recommended_areas.append((m132_area, info))
    
    # Sort by confidence
    recommended_areas.sort(key=lambda x: x[1]['confidence_score'], reverse=True)
    
    return recommended_areas

# Get final recommendations
final_recommendations = get_recommended_visual_m132_areas(fve_overlap_row_norm, visual_areas)

print("\nRECOMMENDED M132 VISUAL AREAS:")
print("=" * 50)
for area, info in final_recommendations:
    print(f"✓ {area:<20} "
          f"Confidence: {info['confidence_score']:.2f} "
          f"Visual Overlap: {info['overlap_percentage']:.1f}%")




RECOMMENDED M132 VISUAL AREAS:
✓ PO_M132              Confidence: 1.00 Visual Overlap: 100.0%
✓ V4t_M132             Confidence: 1.00 Visual Overlap: 100.0%
✓ DP_M132              Confidence: 1.00 Visual Overlap: 99.4%
✓ V4_M132              Confidence: 0.99 Visual Overlap: 98.7%
✓ V3A_M132             Confidence: 0.99 Visual Overlap: 98.0%
✓ V3_M132              Confidence: 0.98 Visual Overlap: 97.4%
✓ V1_M132              Confidence: 0.98 Visual Overlap: 97.3%
✓ MT_M132              Confidence: 0.97 Visual Overlap: 95.1%
✓ V2_M132              Confidence: 0.97 Visual Overlap: 95.1%
✓ PIP_M132             Confidence: 0.95 Visual Overlap: 92.3%
✓ FST_M132             Confidence: 0.92 Visual Overlap: 88.5%
✓ TH_TF_M132           Confidence: 0.92 Visual Overlap: 88.4%
✓ 46v_M132             Confidence: 0.82 Visual Overlap: 74.9%
✓ 9_46v_M132           Confidence: 0.81 Visual Overlap: 73.3%
✓ LIP_M132             Confidence: 0.80 Visual Overlap: 72.1%
✓ 9_46d_M132           Confidence: 0

In [ ]:
final_recommendations

## M132 Visual areas

In [11]:
M132_visual_areas = ['V4t_M132', 'DP_M132', 'V4_M132', 'V3A_M132', 'V3_M132', 
                     'V1_M132', 'MT_M132', 'V2_M132', 'PIP_M132', 'FST_M132', 'TH_TF_M132', 
                     '46v_M132', '9_46v_M132', 'LIP_M132', '9_46d_M132', '46d_M132', 'TEOm_M132', 
                     'TEpd_M132', 'STPi_M132', 'TEO_M132', 'MST_M132', 'STPc_M132', 'V6_M132', 
                     'TEpv_M132', 'TEam-p_M132', '8l_M132', 'VIP_M132', '8r_M132', 'V6A_M132', 
                     'PGa_M132', 'IPa_M132', '45B_M132', '7A_M132', 'TEad_M132', '8m_M132', 
                     'PBc_M132', 'TEam-a_M132', 'MIP_M132', 'STPr_M132']


def extract_with_statistics(data, area_list):
    """
    Extract M132 areas and provide summary statistics
    """
    extracted = {}
    statistics = {}
    
    for area in area_list:
        if area in data:
            extracted[area] = data[area]
            
            # Calculate statistics for this area
            total_overlap = sum(data[area].values())
            num_fve_areas = len(data[area])
            max_overlap = max(data[area].values()) if data[area] else 0
            
            statistics[area] = {
                'total_overlap': total_overlap,
                'num_fve_areas': num_fve_areas,
                'max_overlap': max_overlap,
                'max_fve_area': max(data[area].items(), key=lambda x: x[1]) if data[area] else None
            }
    
    # Print summary
    print(f"Extraction Summary:")
    print(f"Requested areas: {len(area_list)}")
    print(f"Found areas: {len(extracted)}")
    print(f"Missing areas: {len(area_list) - len(extracted)}")
    
    return extracted, statistics

# Extract data with statistics
extracted_data, area_stats = extract_with_statistics(fve_overlap_row_norm, M132_visual_areas)

extracted_data.keys()

Extraction Summary:
Requested areas: 39
Found areas: 39
Missing areas: 0


dict_keys(['V4t_M132', 'DP_M132', 'V4_M132', 'V3A_M132', 'V3_M132', 'V1_M132', 'MT_M132', 'V2_M132', 'PIP_M132', 'FST_M132', 'TH_TF_M132', '46v_M132', '9_46v_M132', 'LIP_M132', '9_46d_M132', '46d_M132', 'TEOm_M132', 'TEpd_M132', 'STPi_M132', 'TEO_M132', 'MST_M132', 'STPc_M132', 'V6_M132', 'TEpv_M132', 'TEam-p_M132', '8l_M132', 'VIP_M132', '8r_M132', 'V6A_M132', 'PGa_M132', 'IPa_M132', '45B_M132', '7A_M132', 'TEad_M132', '8m_M132', 'PBc_M132', 'TEam-a_M132', 'MIP_M132', 'STPr_M132'])

In [12]:
# Load the data in .json file
M132_visual_area_data = {"overlap" : extracted_data}

with open(os.path.join(datapath, 'overlap_data.json'), 'w') as f:
    json.dump(M132_visual_area_data, f, indent=3)


In [ ]:
with open(os.path.join(datapath, 'M132_visual_areas.json'), 'r') as f:
    data = json.load(f)
    
len(data['M132_visual_areas'].keys())